# 관계형 테이블과 키

고객과 주문을 키로 연결하고 조인 행 수가 늘어나는 원인을 확인합니다.

## 실행 환경 준비

In [1]:
# 표 조인과 키 검사를 위해 pandas를 불러옵니다.
import pandas as pd
from pandas.errors import MergeError

print("pandas", pd.__version__)

pandas 3.0.5


## 기본 실행

고객 표의 `customer_id`는 한 고객을 유일하게 구분하는 기본 키입니다. 주문 표의 같은 열은 고객 표를 가리키는 외래 키이며 한 고객에게 여러 주문이 있으므로 반복될 수 있습니다. 두 DataFrame을 입력으로 준비합니다.

In [2]:
# 고객의 기본 키와 주문의 외래 키 역할을 분리해 표를 준비합니다.
customers = pd.DataFrame([
 {"customer_id":"C001", "name":"김하나"},
 {"customer_id":"C002", "name":"이두리"},
])
orders = pd.DataFrame([
 {"order_id":"O101", "customer_id":"C001", "amount":32000},
 {"order_id":"O102", "customer_id":"C002", "amount":18000},
 {"order_id":"O103", "customer_id":"C001", "amount":27000},
])
print(customers.to_string(index=False))
print(orders.to_string(index=False))

customer_id name
       C001  김하나
       C002  이두리
order_id customer_id  amount
    O101        C001   32000
    O102        C002   18000
    O103        C001   27000


## 중간 결과 관찰

DataFrame의 `merge()` 메서드는 두 표를 입력으로 받아 키가 같은 행의 열을 결합한 새 DataFrame을 반환합니다. `on`은 비교할 키, 기본 내부 조인은 양쪽에 모두 있는 키만 남긴다는 뜻입니다. `validate="many_to_one"`은 왼쪽 주문 키의 반복은 허용하고 오른쪽 고객 키는 유일한지 검사합니다.

In [3]:
# many_to_one은 오른쪽 고객 키가 유일한지 검사합니다.
joined = orders.merge(customers, on="customer_id", validate="many_to_one")
print(joined[["order_id","name","amount"]].to_string(index=False))
assert len(joined) == len(orders)

order_id name  amount
    O101  김하나   32000
    O102  이두리   18000
    O103  김하나   27000


## 값 하나를 바꿔 비교

`pd.concat([표1, 표2])`는 표2의 행을 표1 아래에 이어 붙인 새 DataFrame을 반환합니다. 이 함수로 고객 표에 없는 C999 주문을 추가하고 `how="left"`로 왼쪽 조인을 실행합니다. `indicator=True`가 만든 `_merge` 열의 `left_only`와 고객 이름 `NaN`이 같은 행에 나타나는지 확인합니다.

In [4]:
# 부모 고객이 없는 주문은 왼쪽 조인의 표시 열로 발견합니다.
with_orphan = pd.concat([orders, pd.DataFrame([
    {"order_id":"O104", "customer_id":"C999", "amount":9000}
])])
checked = with_orphan.merge(customers, on="customer_id", how="left", indicator=True)
print(checked.loc[checked["_merge"] == "left_only",
                  ["order_id", "customer_id", "name", "_merge"]].to_string(index=False))
assert checked.loc[checked["_merge"] == "left_only", "order_id"].tolist() == ["O104"]

order_id customer_id name    _merge
    O104        C999  NaN left_only


## 실패 재현

`concat`으로 C001 고객을 한 행 더 추가합니다. 일반 조인은 C001 주문을 각각 두 번 결합하므로 행 수뿐 아니라 금액 합계도 증가합니다. 원본 합계와 조인 뒤 합계를 실제 출력으로 비교한 뒤 `many_to_one` 검사가 같은 입력을 거부하는지 확인합니다.

In [5]:
# 중복 고객 키는 주문을 복제하며 관계 검증에서도 거부됩니다.
duplicated = pd.concat([customers, pd.DataFrame([
    {"customer_id":"C001", "name":"김하나(중복)"}
])], ignore_index=True)
wrong = orders.merge(duplicated, on="customer_id")
original_total = int(orders["amount"].sum())
wrong_total = int(wrong["amount"].sum())
print("주문 행:", len(orders), "조인 행:", len(wrong))
print("원본 합계:", original_total, "중복 조인 합계:", wrong_total)
try:
    orders.merge(duplicated, on="customer_id", validate="many_to_one")
except MergeError as error:
    print(type(error).__name__, "중복 키 거부")
else:
    raise AssertionError("중복 키가 거부되지 않았습니다.")
assert len(wrong) == 5 and original_total == 77000 and wrong_total == 136000

주문 행: 3 조인 행: 5
원본 합계: 77000 중복 조인 합계: 136000
MergeError 중복 키 거부


## 수정

`drop_duplicates("customer_id", keep="first")`는 중복 고객 번호에서 먼저 나타난 행만 남긴 새 DataFrame을 반환합니다. `is_unique`는 고객 번호가 모두 유일한지 검사합니다. `fixed.iloc[0]["name"]`은 위치가 첫 번째인 행에서 이름 열을 읽습니다. 수정 뒤 행 수와 금액 합계가 원본과 같아졌는지 확인합니다.

In [6]:
# 교육 데이터는 먼저 온 원본을 유지한다는 규칙으로 중복을 정리합니다.
deduplicated = duplicated.drop_duplicates("customer_id", keep="first")
assert deduplicated["customer_id"].is_unique
fixed = orders.merge(deduplicated, on="customer_id", validate="many_to_one")
fixed_total = int(fixed["amount"].sum())
print("수정 후 조인 행:", len(fixed), "합계:", fixed_total,
      "C001 이름:", fixed.iloc[0]["name"])
assert len(fixed) == 3 and fixed_total == 77000 and fixed.iloc[0]["name"] == "김하나"

수정 후 조인 행: 3 합계: 77000 C001 이름: 김하나
